In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q gensim wget nltk

  Preparing metadata (setup.py) ... done


In [3]:
!pip install -q --upgrade "gensim==4.3.3" "smart-open>=7"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 43.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jup

In [4]:
#basic imports
import warnings
warnings.filterwarnings('ignore')
import os
import wget
import gzip
import shutil
from time import time

#pre-processing imports
import nltk
nltk.download('stopwords')
nltk.download('punkt')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from string import punctuation

#imports related to modeling
import numpy as np
from gensim.models import Word2Vec, KeyedVectors
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


ModuleNotFoundError: No module named 'numpy.char'

In [ ]:
import os
import zipfile
import urllib.request

# Folder dataset
os.makedirs("Data/sentiment labelled sentences", exist_ok=True)

# Download dataset Sentiment Labelled Sentences
url = "https://archive.ics.uci.edu/static/public/331/sentiment+labelled+sentences.zip"
zip_path = "Data/sentiment_labelled_sentences.zip"

if not os.path.exists(zip_path):
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, zip_path)
    print("Download selesai.")

# Extract
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("Data")

# Gabungkan 3 dataset
filenames = [
    'amazon_cells_labelled.txt',
    'imdb_labelled.txt',
    'yelp_labelled.txt'
]

with open('Data/sentiment_sentences.txt', 'w', encoding='utf-8') as outfile:
    for fname in filenames:
        file_path = 'Data/sentiment labelled sentences/' + fname
        
        with open(file_path, 'r', encoding='utf-8') as infile:
            outfile.write(infile.read())

print("File created")

In [ ]:
try:
    from google.colab import files
    
    # upload 'amazon_cells_labelled.txt', 'imdb_labelled.txt' and 'yelp_labelled.txt' present in "sentiment labelled sentences" folder
    uploaded = files.upload()
    
    !mkdir DATAPATH
    !mv -t DATAPATH amazon_cells_labelled.txt imdb_labelled.txt yelp_labelled.txt
    !cat DATAPATH/amazon_cells_labelled.txt DATAPATH/imdb_labelled.txt DATAPATH/yelp_labelled.txt > DATAPATH/sentiment_sentences.txt
    
except ModuleNotFoundError:

    fil = 'sentiment_sentences.txt'

    if not os.path.exists("Data/sentiment_sentences.txt"):
        file = open(os.path.join(path, fil), 'w')
        file.close()
        
        # combined the three files to make sentiment_sentences.txt
        filenames = ['amazon_cells_labelled.txt', 'imdb_labelled.txt', 'yelp_labelled.txt']

        with open('Data/sentiment_sentences.txt', 'w') as outfile:
            for fname in filenames:
                with open('Data/sentiment labelled sentences/' + fname) as infile:
                    outfile.write(infile.read())
        print("File created")
    else:
        print("File already exists")

In [ ]:
#Load the pre-trained word2vec model and the dataset
try:
    
    from google.colab import files
    data_path= "DATAPATH"
    !wget -P DATAPATH https://s3.amazonaws.com/dl4j-distribution/GoogleNews-vectors-negative300.bin.gz
    !gunzip DATAPATH/GoogleNews-vectors-negative300.bin.gz      
    path_to_model = 'DATAPATH/GoogleNews-vectors-negative300.bin'
    training_data_path = "DATAPATH/sentiment_sentences.txt"
    
except ModuleNotFoundError:
    
    data_path= "Data"
    
    if not os.path.exists('GoogleNews-vectors-negative300.bin'):
        if not os.path.exists('../Ch2/GoogleNews-vectors-negative300.bin'):
            if not os.path.exists('../Ch3/GoogleNews-vectors-negative300.bin'):
                wget.download("https://s3.amazonaws.com/dl4j-distribution/GoogleNews-vectors-negative300.bin.gz")

                with gzip.open('GoogleNews-vectors-negative300.bin.gz', 'rb') as f_in:
                    with open('GoogleNews-vectors-negative300.bin', 'wb') as f_out:
                        shutil.copyfileobj(f_in, f_out)

                path_to_model = 'GoogleNews-vectors-negative300.bin'
            else:
                path_to_model = '../Ch3/GoogleNews-vectors-negative300.bin'

        else:
            path_to_model = '../Ch2/GoogleNews-vectors-negative300.bin'
    else:
        path_to_model = 'GoogleNews-vectors-negative300.bin'
        
    training_data_path = os.path.join(data_path, "sentiment_sentences.txt")
    
    
#Load W2V model. This will take some time. 
%time w2v_model = KeyedVectors.load_word2vec_format(path_to_model, binary=True)
print('done loading Word2Vec')

#Read text data, cats.
#the file path consists of tab separated sentences and cats.
texts = []
cats = []
fh = open(training_data_path)
for line in fh:
    text, sentiment = line.split("\t")
    texts.append(text)
    cats.append(sentiment)

In [ ]:
import os
import gzip
import shutil
import wget
from gensim.models import KeyedVectors

data_path = "Data"

path_to_model = os.path.join(
    data_path,
    "GoogleNews-vectors-negative300.bin"
)

training_data_path = os.path.join(
    data_path,
    "sentiment_sentences.txt"
)

# Download model jika belum ada
if not os.path.exists(path_to_model):
    print("Downloading GoogleNews Word2Vec model...")
    
    gz_path = os.path.join(
        data_path,
        "GoogleNews-vectors-negative300.bin.gz"
    )
    
    wget.download(
        "https://s3.amazonaws.com/dl4j-distribution/GoogleNews-vectors-negative300.bin.gz",
        gz_path
    )
    
    print("\nExtracting model...")
    with gzip.open(gz_path, 'rb') as f_in:
        with open(path_to_model, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    
    os.remove(gz_path)

print("Model path:", path_to_model)
print("Training data:", training_data_path)

In [ ]:
# Inspect the model
word2vec_vocab = w2v_model.key_to_index.keys()
word2vec_vocab_lower = [item.lower() for item in word2vec_vocab]

print(len(word2vec_vocab))

In [ ]:
#Inspect the dataset
print(len(cats), len(texts))
print(texts[1])
print(cats[1])

In [ ]:
#preprocess the text.
def preprocess_corpus(texts):
    mystopwords = set(stopwords.words("english"))
    def remove_stops_digits(tokens):
        #Nested function that lowercases, removes stopwords and digits from a list of tokens
        return [token.lower() for token in tokens if token.lower() not in mystopwords and not token.isdigit()
               and token not in punctuation]
    #This return statement below uses the above function to process twitter tokenizer output further. 
    return [remove_stops_digits(word_tokenize(text)) for text in texts]

texts_processed = preprocess_corpus(texts)
print(len(cats), len(texts_processed))
print(texts_processed[1])
print(cats[1])

In [ ]:
# Creating a feature vector by averaging all embeddings for all sentences
def embedding_feats(list_of_lists):
    DIMENSION = 300
    zero_vector = np.zeros(DIMENSION)
    feats = []
    for tokens in list_of_lists:
        feat_for_this =  np.zeros(DIMENSION)
        count_for_this = 0 + 1e-5 # to avoid divide-by-zero 
        for token in tokens:
            if token in w2v_model:
                feat_for_this += w2v_model[token]
                count_for_this +=1
        if(count_for_this!=0):
            feats.append(feat_for_this/count_for_this) 
        else:
            feats.append(zero_vector)
    return feats


train_vectors = embedding_feats(texts_processed)
print(len(train_vectors))

In [ ]:
#Take any classifier (LogisticRegression here, and train/test it like before.
classifier = LogisticRegression(random_state=1234)
train_data, test_data, train_cats, test_cats = train_test_split(train_vectors, cats)
classifier.fit(train_data, train_cats)
print("Accuracy: ", classifier.score(test_data, test_cats))
preds = classifier.predict(test_data)
print(classification_report(test_cats, preds))